# Tesla & GameStop Stock and Revenue Analysis — Final Project

This single notebook consolidates Questions 1–6. Run cells top to bottom in Google Colab. It retrieves current stock data and scrapes revenue tables at runtime. Website markup may change or access may be restricted, so verify outputs before taking screenshots.

1. Tesla stock data (`tesla_data.head()`)
2. Tesla revenue scraping (`tesla_revenue.tail()`)
3. GameStop stock data (`gme_data.head()`)
4. GameStop revenue scraping (`gme_revenue.tail()`)
5. Tesla stock graph using `make_graph()`
6. GameStop stock graph using `make_graph()`

In [ ]:
%pip -q install yfinance pandas requests beautifulsoup4 lxml plotly

In [ ]:
import re
import requests
import pandas as pd
import yfinance as yf
import plotly.express as px
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/124 Safari/537.36"}

def scrape_revenue_table(url, company_name):
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    tables = soup.find_all("table")
    for table in tables:
        try:
            df = pd.read_html(str(table))[0]
            vals = " ".join(str(v).lower() for v in df.astype(str).values.flatten())
            cols = " ".join(str(c).lower() for c in df.columns)
            if "revenue" in vals or "revenue" in cols:
                return df
        except ValueError:
            continue
    all_tables = pd.read_html(response.text)
    if all_tables:
        return all_tables[0]
    raise ValueError(f"No revenue table found for {company_name}; site may have changed or blocked scraping.")

## Question 1 — Use yfinance to Extract Tesla Stock Data

In [ ]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
tesla_data.to_csv("tesla_data.csv", index=False)
tesla_data.head()

## Question 2 — Use Webscraping to Extract Tesla Revenue Data

In [ ]:
tesla_revenue = scrape_revenue_table("https://www.macrotrends.net/stocks/charts/TSLA/tesla/revenue", "Tesla")
tesla_revenue.tail()

## Question 3 — Use yfinance to Extract GameStop Stock Data

In [ ]:
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")
gme_data.reset_index(inplace=True)
gme_data.to_csv("gme_data.csv", index=False)
gme_data.head()

## Question 4 — Use Webscraping to Extract GME Revenue Data

In [ ]:
gme_revenue = scrape_revenue_table("https://www.macrotrends.net/stocks/charts/GME/gamestop/revenue", "GameStop")
gme_revenue.tail()

## Questions 5 and 6 — Plot Tesla and GameStop Stock Graphs

In [ ]:
def make_graph(stock_data, revenue_data, stock):
    date_col = "Date" if "Date" in stock_data.columns else stock_data.columns[0]
    fig = px.line(stock_data, x=date_col, y="Close", title=f"{stock} Historical Share Price",
                  labels={date_col: "Date", "Close": "Share Price (USD)"})
    fig.update_layout(template="plotly_white", title_x=0.5)
    return fig

tesla_graph = make_graph(tesla_data, tesla_revenue, "Tesla")
tesla_graph.show()

In [ ]:
gme_graph = make_graph(gme_data, gme_revenue, "GameStop")
gme_graph.show()

## Submission notes

Run every cell and confirm the tables/graphs display before taking screenshots. For Question 7, save a copy to Google Drive and share the notebook link according to your course instructions. Do not submit screenshots of failed cells as completed results.